### Buffet-Lotterie – Lösung Schritt für Schritt

*33. Bundeswettbewerb Informatik (2014), 1. Runde, Aufgabe 3*

Bei der Endrunde des Bundeswettbewerbs Informatik haben die Teilnehmer es satt, Warteschlangenfutter vor dem großen Buffet im engen Korridor und Opfer der Last-Come-Longest-Hungry-Mentalität zu sein. Stattdessen soll ganz elegant und zivilisiert ausgelost werden, wer als Nächster das Buffet aufsuchen darf.

Die 28 Teilnehmer stellen sich dazu in einem großen Kreis auf und sagen den Satz

> In-for-ma-tik kann uns wei-sen, wer als Nächs-ter kommt zum Spei-sen

wiederholt laut auf. Wie beim Ene-Mene-Muh spricht jeder Teilnehmer nur eine Silbe, dann ist sein rechter Nachbar an der Reihe. Und wer die letzte Silbe des Satzes sagt, ist der Glückliche, der den Kreis verlassen und als Nächster seinen Hunger stillen darf.

Eine Teilnehmerin hat aber Geburtstag. Sie spricht natürlich die allererste Silbe, und als besondere Gunst darf sie, wann immer sie an der Reihe ist, statt einer auch zwei Silben sprechen – wenn sie das denn will. Da ihr Magen knurrt, möchte sie ihren Vorteil dazu nutzen, so schnell wie möglich zum Buffet zu kommen.

#### Aufgabe 3

Schreibe ein Programm, das für eine gegebene Anzahl von Teilnehmern berechnet, wann das Geburtstagskind zwei Silben sprechen soll, um sich den bestmöglichen Platz in der Buffetreihenfolge zu verschaffen.

Dokumentiere die Wirkungsweise deines Programms für verschiedene Teilnehmerzahlen, unter anderem für die oben genannten 28 Teilnehmer.

**Zu den Daten:** Der Satz hat 16 Silben. Beispieldaten gibt es zu dieser Aufgabe nicht. In `eigene_testdaten/beispiele.txt` stehen neun Paare „Teilnehmer Silben“, nämlich die Beispiele aus den Lösungshinweisen.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation:** Man spielt den Vorgang mit dem Computer nach, so wie er in der Aufgabe beschrieben ist. Hier: ein Kreis aus Personen, eine Person nach der anderen sagt Silben, wer die letzte sagt, verlässt den Kreis.
- **Kreis als Liste mit Rest:** Nach dem letzten Platz kommt wieder der erste. Mit dem Rest bei Division (`%`) bleibt eine Platznummer immer im Bereich `0 … n-1`.
- **Alle Möglichkeiten ausprobieren (Brute Force)** und danach **eine Gesetzmäßigkeit suchen**, mit der man nicht mehr probieren muss. Das ist hier der eigentliche Gewinn: Ob Gina einmal mehr oder weniger doppelt spricht, verschiebt die letzte Silbe immer um genau einen Platz, egal in welcher Runde.
- **Zähler für „wie oft war ich schon dran“:** Man muss nicht jede Silbe einzeln durchgehen, es genügt zu rechnen, wie oft der Satz um den Kreis herumläuft (ganzzahlige Division `//`).

**Nützliche Funktionen und Bibliotheken**
- `liste.pop(i)`: nimmt das Element an Platz `i` aus der Liste und gibt es zurück (der Rest rückt auf).
- `a % n` (Rest) und `a // n` (ganzzahlig geteilt).
- `liste.index(x)`: an welchem Platz steht `x`?

## Die Idee in Worten

**Der Kreis.** Wir nummerieren die Stühle von 0 bis `n-1`, Gina sitzt auf Stuhl 0. Die erste Silbe sagt Stuhl 0, die zweite Stuhl 1 und so weiter. Hat der Satz `k` Silben, sagt (ohne Doppelsprechen) Stuhl `(k-1) % n` die letzte Silbe. Diese Person geht zum Buffet, ihr Stuhl verschwindet, alle dahinter rücken einen Platz auf. Die **nächste Runde beginnt bei dem Teilnehmer rechts vom Gegangenen**, und das ist genau der Stuhl mit derselben Nummer.

**Gina darf doppelt sprechen.** Wenn sie in ihrem Zug zwei Silben sagt, ist ab dann jeder weitere Sprecher eine Silbe früher dran. Der Teilnehmer, der die letzte Silbe sagt, sitzt also einen Stuhl weiter links. Zweimal doppelt sprechen heißt zwei Stühle weiter links, unabhängig davon, wann sie das tut. Gina kann so die Stuhlnummer der letzten Silbe um **bis zu so viele Plätze verkleinern, wie sie Gelegenheiten hatte**. Sie ist dran, sobald diese Stuhlnummer nach der Verschiebung bei 0 landet.

**Vorgehen.** Zuerst die Simulation mit einer Liste (so wie die Aufgabe es beschreibt). Dann Alles-Ausprobieren, damit wir eine Referenz haben. Dann das schnelle Verfahren, das nur zwei Zahlen mitführt, und der Vergleich.

## Schritt 1: Einlesen und eine Runde nachspielen

Die Beispiele liegen als Zahlenpaare in einer Datei.

In [ ]:
def lies_beispiele(dateiname):
    liste = []
    for zeile in open(dateiname):
        if zeile.strip():
            n, k = zeile.split()
            liste.append((int(n), int(k)))
    return liste


beispiele = lies_beispiele('eigene_testdaten/beispiele.txt')
print(beispiele)

[(28, 16), (16, 16), (15, 16), (55, 16), (28, 32), (10, 8), (5, 17), (280, 19), (9, 312)]


Eine **Runde** spielen wir Silbe für Silbe nach. Der Kreis ist eine Liste von Teilnehmern (Gina ist die Zahl `0`), `start` ist der Platz, bei dem die Runde beginnt. Gina darf bei ihren ersten `doppelt` Gelegenheiten zwei Silben sagen. Zwei Silben sagt sie nur, wenn danach der Satz noch nicht zu Ende wäre, also wenn mindestens zwei Silben übrig sind. Die Funktion gibt den Platz des Teilnehmers zurück, der die letzte Silbe sagt, und wie viele Doppelungen Gina dabei benutzt hat.

In [ ]:
GINA = 0


def runde(kreis, start, k, doppelt):
    n = len(kreis)
    platz = start
    silben = 0                    # so viele Silben des Satzes sind schon gesagt
    genutzt = 0
    while True:
        if kreis[platz] == GINA and genutzt < doppelt and silben <= k - 2:
            silben = silben + 2
            genutzt = genutzt + 1
        else:
            silben = silben + 1
        if silben == k:
            return platz, genutzt
        platz = (platz + 1) % n


kreis = list(range(28))
print(runde(kreis, 0, 16, 0), runde(kreis, 0, 16, 1))

(15, 0) (14, 1)


Ohne Doppelsprechen sagt Stuhl 15 die letzte Silbe (Stühle 0 bis 15 sind 16 Silben). Mit einer Doppelung rutscht das auf Stuhl 14 vor.

## Schritt 2: Die ganze Lotterie mit festem `d`

Wir spielen so lange Runden, bis alle am Buffet waren. Gina sagt in ihren ersten `d` Gelegenheiten zwei Silben. Die Funktion `lotterie` gibt die Reihenfolge zurück, in der die Teilnehmer ans Buffet dürfen. Nach `pop` rückt der Rest auf, die nächste Runde beginnt am selben Platz (und am Ende der Liste wieder bei 0, deshalb `% len`).

In [ ]:
def lotterie(n, k, d):
    kreis = list(range(n))
    start = 0
    uebrig = d                              # so viele Doppelungen darf Gina noch machen
    reihenfolge = []
    while len(kreis) > 0:
        platz, genutzt = runde(kreis, start, k, uebrig)
        uebrig = uebrig - genutzt
        reihenfolge.append(kreis.pop(platz))
        if len(kreis) > 0:
            start = platz % len(kreis)
    return reihenfolge


def ginas_platz(n, k, d):
    return lotterie(n, k, d).index(GINA) + 1


for d in range(5):
    print(d, 'mal doppelt: Gina kommt als', ginas_platz(28, 16, d), '. dran')

0 mal doppelt: Gina kommt als 19 . dran
1 mal doppelt: Gina kommt als 17 . dran
2 mal doppelt: Gina kommt als 8 . dran
3 mal doppelt: Gina kommt als 28 . dran
4 mal doppelt: Gina kommt als 27 . dran


Ohne Doppelsprechen kommt Gina als 19. dran, mit einer Doppelung als 17., mit zwei Doppelungen als **8.** Mit drei Doppelungen wird es sogar schlechter (28. = Gina geht als Letzte), weil unsere Funktion sie dann auch wirklich dreimal doppelt sprechen lässt; Gina muss also genau die richtige Zahl wählen. Wir suchen den **besten Platz über alle `d`**. Probieren wir `d` von 0 bis 10 aus (bei gleichem Platz zählt das kleinere `d`):

In [ ]:
def beste_d_probieren(n, k, d_max):
    bestes_p = ginas_platz(n, k, 0)
    bestes_d = 0
    for d in range(1, d_max + 1):
        p = ginas_platz(n, k, d)
        if p < bestes_p:
            bestes_p = p
            bestes_d = d
    return bestes_p, bestes_d


print(beste_d_probieren(28, 16, 10))

(8, 2)


Das ist das Ergebnis für den Satz der Aufgabe: Gina kommt als **8.** dran, wenn sie **zweimal** doppelt spricht. Ob `d_max = 10` immer reicht, wissen wir aber nicht, und wir wissen auch nicht, ob die Strategie „doppelt in den ersten Gelegenheiten“ wirklich die beste ist. Dafür brauchen wir Alles-Ausprobieren und später die Begründung.

## Schritt 3: Alles ausprobieren (Referenz für kleine Fälle)

Hier spielen wir **jede** Entscheidung durch: Bei jeder Gelegenheit sagt Gina einmal oder zweimal. Eine Runde liefert dann nicht einen, sondern mehrere mögliche letzte Sprecher, jeweils mit der kleinsten dafür nötigen Zahl von Doppelungen (ein Wörterbuch `Platz → Doppelungen`). Für jeden möglichen Ausgang spielen wir die nächste Runde weiter. Das wird schnell sehr viel. Damit es für kleine Kreise und für die 28 Teilnehmer reicht, merkt sich die Funktion Ergebnisse im Wörterbuch `gemerkt`: Wie es weitergeht, hängt nur von der Zahl der Teilnehmer und von Ginas Abstand zum Start ab.

In [ ]:
def moegliche_enden(kreis, start, k):
    n = len(kreis)
    enden = {}

    def sprechen(platz, silben, doppelt):
        # platz: wer ist als Nächster dran, silben: so viele Silben sind schon gesagt
        if silben == k:
            letzter = (platz - 1) % n
            if letzter not in enden or doppelt < enden[letzter]:
                enden[letzter] = doppelt
            return
        if kreis[platz] == GINA and silben <= k - 2:
            sprechen((platz + 1) % n, silben + 2, doppelt + 1)
        sprechen((platz + 1) % n, silben + 1, doppelt)

    sprechen(start, 0, 0)
    return enden


gemerkt = {}


def probiere_alles(kreis, start, k):
    # Ergebnis: (Platz in der Reihenfolge, Zahl der Doppelungen), kleinste Werte zuerst.
    # Wie es weitergeht, hängt nur von der Zahl der Teilnehmer und Ginas Abstand zum Start ab: merken (Wörterbuch).
    schluessel = (len(kreis), k, (kreis.index(GINA) - start) % len(kreis))
    if schluessel in gemerkt:
        return gemerkt[schluessel]
    bestes = None
    for platz, doppelt in moegliche_enden(kreis, start, k).items():
        if kreis[platz] == GINA:
            ergebnis = (1, doppelt)
        else:
            rest = kreis[:platz] + kreis[platz + 1:]
            if len(rest) == 0:
                continue
            p, d = probiere_alles(rest, platz % len(rest), k)
            ergebnis = (p + 1, d + doppelt)
        if bestes is None or ergebnis < bestes:
            bestes = ergebnis
    gemerkt[schluessel] = bestes
    return bestes


print(probiere_alles(list(range(28)), 0, 16))
print(probiere_alles(list(range(10)), 0, 8))

(8, 2)
(4, 4)


Auch Alles-Ausprobieren liefert für 28 Teilnehmer **Platz 8 mit zweimal Doppelsprechen**, und für 10 Teilnehmer mit 8 Silben **Platz 4 mit viermal**. (Bei gleichem Platz steht die kleinere Zahl von Doppelungen vorn.) Die erste Zahl ist unser Maßstab: Kein Verfahren kann Gina einen besseren Platz verschaffen.

Der Aufwand wächst aber schnell: In jeder Runde, in der Gina mehrmals dran ist, verzweigt sich der Ablauf. Bei vielen Teilnehmern und kurzen Sätzen (oder umgekehrt, wie bei 9 Teilnehmern und 312 Silben) geht das nicht mehr.

## Schritt 4: Das schnelle Verfahren

Wir führen nur zwei Zahlen mit, **ohne** eine Liste von Personen:
- `i`: Stuhlnummer des Teilnehmers, der die letzte Silbe sagt (und in der nächsten Runde die erste). Wir beginnen mit `i = 0` (Gina sagt die erste Silbe). Pro Runde kommen `k - 1` Plätze dazu, denn die letzte Silbe sagt der Teilnehmer `k - 1` Plätze nach dem ersten.
- `c`: Zahl der Gelegenheiten, bei denen Gina zwei Silben sagen konnte. Läuft der Satz einmal ganz um den Kreis (oder mehrfach, wenn nur noch wenige da sind), ist Gina jedes Mal wieder dran. `c` wächst um `i // n`, wenn `i` über den Kreis hinausläuft.

Läuft `i` über `n - 1` hinaus, nehmen wir den Rest `i % n`. Nach jeder Runde geht ein Teilnehmer, `n` wird um 1 kleiner. Dann prüfen wir, ob Gina durch Doppelsprechen den Index auf 0 verringern kann: Das geht, wenn `i <= c`, und sie muss genau `i` Mal doppelt sprechen (egal bei welchen Gelegenheiten). Sie ist dann als `anzahl - n` an der Reihe, wobei `anzahl` die Zahl aller Teilnehmer ist und `n` die Zahl nach dem Abgang.

Warum ist das optimal? In allen früheren Runden war `i > c`: Gina kann den Index pro Gelegenheit nur um 1 verringern, hatte also insgesamt höchstens `c` Verringerungen zur Verfügung und kam nicht auf 0.

In [ ]:
def buffet(anzahl, silben):
    n = anzahl
    i = 0
    c = 1
    while True:
        i = i + silben - 1
        if i >= n:                      # Gina kommt wieder an die Reihe
            c = c + i // n
            i = i % n
        n = n - 1                       # ein Stuhl wird entfernt
        if i <= c:
            return anzahl - n, i        # Platz p, Zahl der Doppelungen


for anzahl, silben in beispiele:
    p, d = buffet(anzahl, silben)
    print(anzahl, 'Personen,', silben, 'Silben: Gina bekommt Platz', p, ', wenn sie', d, 'mal zwei Silben sagt.')

28 Personen, 16 Silben: Gina bekommt Platz 8 , wenn sie 2 mal zwei Silben sagt.
16 Personen, 16 Silben: Gina bekommt Platz 2 , wenn sie 0 mal zwei Silben sagt.
15 Personen, 16 Silben: Gina bekommt Platz 1 , wenn sie 0 mal zwei Silben sagt.
55 Personen, 16 Silben: Gina bekommt Platz 10 , wenn sie 3 mal zwei Silben sagt.
28 Personen, 32 Silben: Gina bekommt Platz 5 , wenn sie 1 mal zwei Silben sagt.
10 Personen, 8 Silben: Gina bekommt Platz 4 , wenn sie 4 mal zwei Silben sagt.
5 Personen, 17 Silben: Gina bekommt Platz 1 , wenn sie 1 mal zwei Silben sagt.
280 Personen, 19 Silben: Gina bekommt Platz 78 , wenn sie 5 mal zwei Silben sagt.
9 Personen, 312 Silben: Gina bekommt Platz 1 , wenn sie 5 mal zwei Silben sagt.


Das sind genau die neun Ergebnisse aus den Lösungshinweisen. Die Schleife läuft höchstens `n` Mal, auch bei 280 Teilnehmern.

## Schritt 5: Stimmt das Verfahren? Probe mit der Simulation und mit Alles-Ausprobieren

Zwei Proben:
1. **Simulation:** Mit dem berechneten `d` spielen wir die Lotterie mit der Liste durch. Gina muss dann tatsächlich als `p`-te dran sein.
2. **Alles-Ausprobieren:** Für kleine Fälle muss das Ergebnis des schnellen Verfahrens gleich dem der Referenz sein (Platz **und** Zahl der Doppelungen).

In [ ]:
for anzahl, silben in beispiele:
    p, d = buffet(anzahl, silben)
    p_sim = ginas_platz(anzahl, silben, d)
    print(anzahl, silben, ': schnell', (p, d), ', Simulation mit d =', d, ':', p_sim, ', ok' if p_sim == p else ', FEHLER')

28 16 : schnell (8, 2) , Simulation mit d = 2 : 8 , ok
16 16 : schnell (2, 0) , Simulation mit d = 0 : 2 , ok
15 16 : schnell (1, 0) , Simulation mit d = 0 : 1 , ok
55 16 : schnell (10, 3) , Simulation mit d = 3 : 10 , ok
28 32 : schnell (5, 1) , Simulation mit d = 1 : 5 , ok
10 8 : schnell (4, 4) , Simulation mit d = 4 : 4 , ok
5 17 : schnell (1, 1) , Simulation mit d = 1 : 1 , ok
280 19 : schnell (78, 5) , Simulation mit d = 5 : 78 , ok
9 312 : schnell (1, 5) , Simulation mit d = 5 : 1 , ok


In [ ]:
geprueft = 0
for n in range(1, 15):
    for k in range(2, 21):
        if probiere_alles(list(range(n)), 0, k) != buffet(n, k):
            print('Unterschied bei', n, k)
        geprueft = geprueft + 1
print(geprueft, 'Kombinationen verglichen')

266 Kombinationen verglichen


Es gibt keinen Unterschied: Das schnelle Verfahren liefert in allen 266 Fällen (Teilnehmerzahl 1 bis 14, Satzlänge 2 bis 20) dasselbe wie Alles-Ausprobieren. (Der Generator dieses Notebooks hat außerdem alle Fälle bis 25 Teilnehmer und 29 Silben verglichen.)

## Schritt 6: Wann soll Gina doppelt sprechen? Der Ablauf bei 28 Teilnehmern

Die Aufgabe will wissen, **wann** sie zwei Silben sprechen soll. Nach der Überlegung oben ist das egal, solange es `d` Mal geschieht. Am einfachsten tut sie es in den **ersten** `d` Gelegenheiten. Wir schreiben den Ablauf für 28 Teilnehmer und 16 Silben auf: pro Runde, wer geht und wie viele Doppelungen Gina in dieser Runde nutzt.

In [ ]:
def ablauf(n, k, d):
    kreis = list(range(n))
    start = 0
    uebrig = d
    runde_nr = 1
    while True:
        platz, genutzt = runde(kreis, start, k, uebrig)
        uebrig = uebrig - genutzt
        wer = 'Gina' if kreis[platz] == GINA else 'Teilnehmer ' + str(kreis[platz])
        print('Runde', runde_nr, ':', wer, 'geht, Gina spricht', genutzt, 'mal doppelt')
        if kreis[platz] == GINA:
            return
        kreis.pop(platz)
        start = platz % len(kreis)
        runde_nr = runde_nr + 1


ablauf(28, 16, 2)

Runde 1 : Teilnehmer 14 geht, Gina spricht 1 mal doppelt
Runde 2 : Teilnehmer 1 geht, Gina spricht 1 mal doppelt
Runde 3 : Teilnehmer 18 geht, Gina spricht 0 mal doppelt
Runde 4 : Teilnehmer 7 geht, Gina spricht 0 mal doppelt
Runde 5 : Teilnehmer 25 geht, Gina spricht 0 mal doppelt
Runde 6 : Teilnehmer 16 geht, Gina spricht 0 mal doppelt
Runde 7 : Teilnehmer 8 geht, Gina spricht 0 mal doppelt
Runde 8 : Gina geht, Gina spricht 0 mal doppelt


Gina nutzt ihre beiden Doppelungen in den ersten beiden Runden (zuerst die Runde 1 mit ihrer ersten Silbe), danach gibt es keine Doppelung mehr, und in Runde 8 geht sie selbst. Zum Vergleich: ohne Doppelsprechen käme sie als 19. dran (Schritt 2, `d = 0`).

## Schritt 7: Die Beispiele und andere Teilnehmerzahlen

Zuerst die Beispiele aus den Hinweisen mit der kleinsten Zahl Doppelungen (alle Zahlen stimmen mit den Hinweisen überein):

| Teilnehmer | Silben | Platz p | Doppelungen |
|---|---|---|---|
| 28 | 16 | 8 | 2 |
| 16 | 16 | 2 | 0 |
| 15 | 16 | 1 | 0 |
| 55 | 16 | 10 | 3 |
| 28 | 32 | 5 | 1 |
| 10 | 8 | 4 | 4 |
| 5 | 17 | 1 | 1 |
| 280 | 19 | 78 | 5 |
| 9 | 312 | 1 | 5 |

Der Satz aus der Aufgabe hat 16 Silben. Für die Teilnehmerzahlen von 1 bis 40 sieht der beste Platz so aus:

In [ ]:
for n in range(1, 41):
    p, d = buffet(n, 16)
    print('n =', n, ': Platz', p, ', doppelt', d, 'mal')

n = 1 : Platz 1 , doppelt 0 mal
n = 2 : Platz 1 , doppelt 1 mal
n = 3 : Platz 1 , doppelt 0 mal
n = 4 : Platz 1 , doppelt 3 mal
n = 5 : Platz 1 , doppelt 0 mal
n = 6 : Platz 1 , doppelt 3 mal
n = 7 : Platz 1 , doppelt 1 mal
n = 8 : Platz 2 , doppelt 1 mal
n = 9 : Platz 3 , doppelt 6 mal
n = 10 : Platz 2 , doppelt 2 mal
n = 11 : Platz 4 , doppelt 5 mal
n = 12 : Platz 3 , doppelt 2 mal
n = 13 : Platz 1 , doppelt 2 mal
n = 14 : Platz 1 , doppelt 1 mal
n = 15 : Platz 1 , doppelt 0 mal
n = 16 : Platz 2 , doppelt 0 mal
n = 17 : Platz 4 , doppelt 1 mal
n = 18 : Platz 6 , doppelt 2 mal
n = 19 : Platz 8 , doppelt 3 mal
n = 20 : Platz 5 , doppelt 5 mal
n = 21 : Platz 4 , doppelt 3 mal
n = 22 : Platz 4 , doppelt 0 mal
n = 23 : Platz 3 , doppelt 2 mal
n = 24 : Platz 3 , doppelt 0 mal
n = 25 : Platz 6 , doppelt 3 mal
n = 26 : Platz 10 , doppelt 6 mal
n = 27 : Platz 5 , doppelt 2 mal
n = 28 : Platz 8 , doppelt 2 mal
n = 29 : Platz 2 , doppelt 2 mal
n = 30 : Platz 2 , doppelt 1 mal
n = 31 : Platz 2 ,

Bei 15 Teilnehmern kommt Gina sofort dran, weil `16 - 1 = 15` genau einmal um den Kreis geht: die 16. Silbe fällt wieder auf Gina, sie ist also schon in der ersten Runde die Letzte. Auch bei vielen Teilnehmern ist die Rechnung schnell:

In [ ]:
for n in (100, 1000, 100000):
    p, d = buffet(n, 16)
    print(n, 'Teilnehmer: Platz', p, ', doppelt', d, 'mal')

100 Teilnehmer: Platz 23 , doppelt 2 mal
1000 Teilnehmer: Platz 276 , doppelt 1 mal
100000 Teilnehmer: Platz 32107 , doppelt 4 mal


Die Schleife läuft höchstens so oft, wie Teilnehmer da sind, die Laufzeit wächst also nur **linear**. Die Simulation mit der Liste (`pop`) wäre bei 100 000 Teilnehmern viel langsamer. Das Verfahren funktioniert auch für andere Sätze: Es kommt nur auf die Zahl der Silben `k` an (siehe die Beispiele mit 8, 17, 19, 32 und 312 Silben).

## Zum Schluss: Geht das auch mit MiniZinc?

Ja, wenn man das Spiel **Runde für Runde** als Modell aufschreibt. Wir lassen Gina in jeder Runde `r` eine Zahl `m[r]` von Doppelungen wählen und lassen MiniZinc die Runde `p` suchen, in der sie selbst die letzte Silbe sagt, möglichst klein. Der Aufbau folgt der Überlegung aus der Idee: `g[r]` ist Ginas Platz in der Runde `r`, vom ersten Sprecher aus gezählt, `l` der Platz des Sprechers der letzten Silbe (`K - 1 - m[r]`, modulo Kreisgröße). Solange `l` nicht Ginas Platz ist, geht jemand anderes, und Gina rückt auf (`g[r+1]`); in Runde `p` ist `l = g[r]`. Wie oft Gina in einer Runde höchstens doppelt sprechen kann (`t`), rechnet das Modell aus ihrem Platz aus. Zwei Hinweise für Anfänger: `div` und `mod` sind ganzzahlige Division und Rest, und bei negativen Zahlen ist `mod` in MiniZinc negativ, deshalb wird jeweils `+ n` addiert. Die **Suchstrategie** `int_search([p] ++ m, …)` sagt dem Solver, er soll zuerst `p` festlegen (von klein nach groß) und dann die `m`.

```minizinc
int: N;   % Teilnehmer
int: K;   % Silben des Satzes
var 1..N: p;                       % Runde, in der Gina zum Buffet darf
array[1..N] of var 0..K: m;        % wie oft Gina in Runde r doppelt spricht
array[1..N+1] of var 0..N: g;      % Ginas Platz in Runde r, vom ersten Sprecher an gezählt

constraint g[1] = 0;
constraint forall(r in 1..N)(
  let { int: n = N - r + 1;                                         % so viele sind noch im Kreis
        % so oft kann Gina doppelt sprechen, wenn sie es jedes Mal tut:
        var 0..K: t = if g[r] <= K - 2 then (K - 2 - g[r]) div (n + 1) + 1 else 0 endif;
        % wer sagt die letzte Silbe? (+ n, weil mod bei negativen Zahlen negativ bleibt)
        var 0..n-1: l = (K - 1 - m[r] + n) mod n } in
  m[r] <= t /\
  if r < p then l != g[r] /\ g[r+1] = (g[r] - l - 1 + n) mod n      % Gina bleibt, der Kreis rückt auf
  elseif r = p then l = g[r]                                        % Gina geht
  else m[r] = 0 /\ g[r+1] = 0 endif);                               % danach ist alles egal

solve :: int_search([p] ++ m, input_order, indomain_min) minimize p;
output ["p = \(p)\nDoppelungen = \(sum(r in 1..N)(if r <= p then m[r] else 0 endif))\n"];
```

Die Daten für das Beispiel mit 28 Teilnehmern (`buffet.dzn`), alle anderen Beispiele entsprechend:

```minizinc
N = 28;
K = 16;
```

**Ausgabe** (Gecode, letzte und damit beste Lösung):

```
p = 8
Doppelungen = 2
```

Das ist wieder Platz 8 mit zwei Doppelungen. **Messung bei allen neun Beispielen** (Gecode, jeweils unter einer Sekunde):

| Teilnehmer | Silben | Platz p (MiniZinc) | Platz p (Python) | Doppelungen (MiniZinc) | Doppelungen (Python) |
|---|---|---|---|---|---|
| 28 | 16 | 8 | 8 | 2 | 2 |
| 16 | 16 | 2 | 2 | 0 | 0 |
| 15 | 16 | 1 | 1 | 0 | 0 |
| 55 | 16 | 10 | 10 | 3 | 3 |
| 28 | 32 | 5 | 5 | 1 | 1 |
| 10 | 8 | 4 | 4 | 4 | 4 |
| 5 | 17 | 1 | 1 | 1 | 1 |
| 280 | 19 | 78 | 78 | 5 | 5 |
| 9 | 312 | 1 | 1 | 5 | 5 |

Die Plätze sind in allen neun Fällen gleich. Bei den Doppelungen kann MiniZinc eine **andere** Zahl nennen als Python, weil es nur den Platz `p` minimiert und danach irgendeine passende Verteilung der Doppelungen ausgibt, während Python die **kleinste** Zahl von Doppelungen bestimmt (die Fragestellung ist also eine andere). Jede ausgegebene Zahl führt aber, wie die Probe mit `ginas_platz` zeigt, wirklich zu Platz `p`.

**Grenzen:** Ohne die Suchstrategie fand Gecode schon bei 28 Teilnehmern nichts (er legt zuerst `m` fest und bemerkt Widersprüche erst spät). Chuffed bricht bei diesem Modell mit einer Fehlermeldung ab (`div` und `mod` auf Variablen, deren Wertebereich negative Zahlen enthält). Und das Modell hat `N` Runden mit jeweils eigenen Variablen, es rechnet im Grunde dasselbe wie die Schleife in Python, ist aber umständlicher. Für diese Aufgabe ist Python die bessere Wahl; MiniZinc zeigt hier vor allem, wie man eine Simulation in Bedingungen übersetzt.

## Für Neugierige

- **Freunde mitnehmen:** Die Hinweise schlagen vor: Für Gina ist egal, in welchen ihrer Gelegenheiten sie doppelt spricht, für die anderen Teilnehmer nicht. Vielleicht kann sie erreichen, dass auch ihre besten Freunde früh dran sind? Dafür müsste man mit `lotterie` die verschiedenen Verteilungen der Doppelungen vergleichen.
- **Ene-Mene-Muh und Josephus:** Ohne Gina ist die Aufgabe das bekannte **Josephus-Problem**: Jede `k`-te Person scheidet aus. Das Verfahren rechnet den Index mit Rest weiter, ohne eine Liste zu führen.
- **Andere Sätze:** Das Verfahren braucht nur die Silbenzahl `k`. Probiere `buffet(28, 17)` oder einen sehr langen Satz.
- **Mehr Vorteile:** Was wäre, wenn Gina auch drei Silben sagen dürfte? Dann verschiebt jede Dreifach-Silbe die letzte Silbe um 2 Plätze. Wie müsste man `c` anpassen?